# Trabalho Final - Redes Neurais Profundas

## Analise Arquitetural do Congelamento de Camadas na Mitigacao de *Domain Shift* e *Language Shift* em Transformers Multilingues

**UFG / INF - Redes Neurais Profundas**

Este notebook reune, em ordem, todo o percurso do trabalho: a pergunta de pesquisa, a fundamentacao teorica, as hipoteses, o desenho experimental e a execucao ponta a ponta - engenharia de dados, arquitetura, treino e analise estatistica. O fio condutor e uma unica pergunta: **ate que ponto congelar seletivamente camadas do XLM-RoBERTa durante o *fine-tuning* preserva a robustez do modelo quando a distribuicao de teste muda** - seja de **dominio** (Eletronicos para Beleza), seja de **idioma**, do mais proximo (Portugues) ao tipologicamente **distante** (Japones, Mandarim).

O notebook e auto-contido e reproduzivel. As etapas de dados, arquitetura e treino exigem GPU (Colab/Kaggle); a etapa de analise roda sem GPU, com a base de resultados embutida no proprio notebook.

## 1. Pergunta de pesquisa e objetivos

Modelos multilingues como o XLM-RoBERTa sao pre-treinados em dezenas de idiomas e, por isso, prometem **transferencia *zero-shot cross-lingual***: treinar a tarefa em um idioma e aplicar em outro sem novos rotulos. Na pratica, porem, o *fine-tuning* completo pode especializar o modelo no idioma e no dominio de treino, corroendo essa transferencia.

**Pergunta central.** O congelamento seletivo de camadas durante o *fine-tuning* melhora a robustez do XLM-RoBERTa sob deslocamento de distribuicao - de dominio e de idioma - e, em particular, a transferencia para idiomas **tipologicamente distantes** do idioma de treino?

**Objetivos especificos.**

1. Quantificar a degradacao de desempenho ao mudar o **dominio** (Eletronicos para Beleza, mesmo idioma).
2. Quantificar a degradacao ao mudar o **idioma**, partindo de um par **proximo** (Ingles para Portugues) e tendo como horizonte um par **distante** (Ingles para Japones/Mandarim).
3. Testar se estrategias de congelamento (congelar a base lexico-sintatica ou o topo semantico) **mitigam** essas quedas, e medir a significancia estatistica dos efeitos.

A tarefa-alvo e **classificacao binaria de sentimento** (positivo/negativo) em avaliacoes de produtos. O modelo e sempre treinado em **Ingles/Eletronicos** e avaliado em celulas que isolam cada tipo de deslocamento.

## 2. Fundamentacao: por que congelar camadas pode importar

A literatura de interpretabilidade de Transformers (a chamada *BERTology*) sugere uma **hierarquia funcional** entre as camadas do encoder:

- **Camadas inferiores (0-5):** processamento lexico e sintatico - representacoes mais ligadas a forma e ao idioma. E nessas camadas que mora boa parte do **alinhamento multilingue** aprendido no pre-treino.
- **Camadas superiores (6-11):** abstracoes semanticas, mais alinhadas a tarefa especifica de *fine-tuning*.

Dessa hierarquia nasce a intuicao do trabalho: **congelar as camadas certas pode preservar exatamente a parte do modelo responsavel pela generalizacao**. Congelar a base preservaria o alinhamento multilingue (bom para mudar de idioma); congelar o topo evitaria a especializacao excessiva no dominio de treino (bom para mudar de dominio).

A robustez a idiomas distantes e o caso mais exigente desta hipotese: e onde o alinhamento multilingue de baixo nivel teria o maior valor a preservar - e, portanto, onde o efeito do congelamento da base deveria aparecer com mais forca, se existir.

## 3. Hipoteses

A partir da hierarquia funcional, formulamos duas hipoteses falseaveis:

- **H1 (*Language Shift*).** Congelar as camadas **inferiores** (*Freeze Lower*, **C2**) preserva o alinhamento multilingue do pre-treino e **reduz a queda** de desempenho ao inferir em outro idioma um modelo treinado em Ingles - efeito que deveria ser tanto maior quanto **mais distante** o idioma de teste.
- **H2 (*Domain Shift*).** Congelar as camadas **superiores** (*Freeze Upper*, **C3**) reduz a especializacao no dominio de Eletronicos e **diminui a queda** ao avaliar em Beleza.

Estrategias de congelamento comparadas (encoder de 12 camadas):

| Config | Estrategia | Camadas treinaveis |
|--------|-----------|--------------------|
| **C1** | *Full Fine-Tuning* | todas (encoder + *head*) |
| **C2** | *Freeze Lower* | congela embeddings + camadas 0-5; treina 6-11 + *head* |
| **C3** | *Freeze Upper* | congela camadas 6-11; treina embeddings + 0-5 + *head* |
| **C4** | *Frozen Encoder* | congela todo o encoder; treina so a *head* |

A *classification head* nasce do zero e e **sempre treinavel**.

## 4. Desenho experimental

O modelo e **sempre treinado em S1 = Ingles/Eletronicos**. A avaliacao usa celulas que isolam cada eixo de deslocamento:

| Celula | Conjunto | Idioma / Dominio | Tipo de deslocamento |
|--------|----------|------------------|----------------------|
| **T1** | S1_val | EN / Eletronicos | nenhum (*baseline in-domain, in-language*) |
| **T2** | S2 | EN / Beleza | **Domain Shift** |
| **T3** | S3 | PT / Eletronicos | **Language Shift proximo** (Portugues, *ground-truth* B2W) |
| **T4** | S4 | PT / Beleza | Dominio + Idioma (combinado) |
| **T(dist)** | MARC | JA / ZH | **Language Shift distante** (horizonte do estudo - Secao 9) |

**Metrica.** F1-macro (robusta a desbalanceamento), com acuracia e F1 por classe como apoio.

**Replicacao.** Cada configuracao e treinada com **multiplas seeds independentes**. As seeds afetam tanto a ordem dos lotes quanto a inicializacao da *head* (critico na C4, que so treina a *head*). Os resultados consolidados deste notebook agregam as execucoes independentes dos integrantes do grupo, somando **6 a 8 seeds por configuracao**.

**Testes estatisticos.** Cada estrategia e comparada a baseline C1 com o **t de Welch** (primario, variancias desiguais), **Mann-Whitney U** (confirmacao nao-parametrica) e **Cohen's d** (tamanho de efeito). Limiar de significancia **p < 0.10**.

Os hiperparametros sao **identicos entre as configuracoes** de proposito: isso isola a unica variavel de interesse - o congelamento.

## 5. Mapa do notebook

| Etapa | Conteudo | Requer GPU |
|-------|----------|:----------:|
| **0** | Exploracao previa de dominios candidatos (B2W) | nao |
| **1** | Engenharia de dados: filtragem hibrida, balanceamento e auditoria | sim (auditoria) |
| **2** | Arquitetura do XLM-RoBERTa e logica de congelamento C1-C4 | sim |
| **3** | Treino dos modelos e extracao de metricas | sim |
| **4** | Analise estatistica consolidada (multi-seed) - cenarios solidos | nao |
| **9** | Fronteira: *Language Shift* distante (proxima execucao) | sim |

As Etapas 0-3 documentam e executam a construcao do estudo. A Etapa 4 e auto-contida (base embutida) e entrega os resultados solidos. A Etapa 9 detalha o experimento de lingua distante e o ajuste necessario para realiza-lo de forma valida.

---
## Etapa 0 - Exploracao previa de dominios candidatos

Antes de fixar o par de dominios, e preciso garantir que **ambas as classes (positivo e negativo) tenham volume suficiente** no lado portugues (B2W), que e o mais escasso. A escolha do par recaiu sobre **Eletronicos x Beleza**: vocabulario distinto, classe negativa robusta nos dois dominios e volume adequado. A celula abaixo mede o *gargalo* (a classe minoritaria) por categoria, criterio que orientou a decisao.

In [ ]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Download do B2W
CAMINHO_B2W = "/content/b2w.csv"
URL_B2W = "https://raw.githubusercontent.com/americanas-tech/b2w-reviews01/main/B2W-Reviews01.csv"

print(" Baixando B2W Reviews...")
if not os.path.exists(CAMINHO_B2W):
    if not os.path.exists("/content"):
        os.makedirs("/content", exist_ok=True)
    df_b2w = pd.read_csv(URL_B2W, low_memory=False)
    df_b2w.to_csv(CAMINHO_B2W, index=False)
else:
    df_b2w = pd.read_csv(CAMINHO_B2W, low_memory=False)
    
print(f"OK Base B2W carregada com {df_b2w.shape[0]:,} avaliações.")

In [ ]:
# 2. Mapeamento de Rótulos
def estrela_para_sentimento(nota):
    try:
        n = float(nota)
    except (TypeError, ValueError):
        return None
    if n <= 2: return 'Negativo'  # 0 no script final, mas string ajuda na visualizacao aqui
    if n >= 4: return 'Positivo'  # 1 no script final
    return None

df_b2w['label'] = df_b2w['overall_rating'].apply(estrela_para_sentimento)
df_b2w_validos = df_b2w.dropna(subset=['label']).copy()

print(f"- Após descartar notas 3 (neutras) e nulas, sobraram {df_b2w_validos.shape[0]:,} avaliações.")
print(df_b2w_validos['label'].value_counts())

In [ ]:
# 3. Agrupamento por Categoria (site_category_lv1)
# Vamos descobrir quais categorias têm uma boa representação nas DUAS classes.

# Tabela Pivot: Categorias x Sentimentos
tabela_categorias = pd.crosstab(df_b2w_validos['site_category_lv1'], df_b2w_validos['label'])

# O Gargalo é o mínimo entre Positivos e Negativos, 
# já que o modelo precisará fazer undersampling (balanceamento) por classe
tabela_categorias['Gargalo (Balanceado)'] = tabela_categorias.min(axis=1)
tabela_categorias['Total'] = tabela_categorias['Negativo'] + tabela_categorias['Positivo']

# Ordenando pelo gargalo (aquelas que nos permitirão o maior número final equilibrado)
tabela_categorias = tabela_categorias.sort_values(by='Gargalo (Balanceado)', ascending=False)

print(" TOP 15 Categorias mais 'seguras' para serem escolhidas como domínio (ordenadas pelo gargalo):\n")
display(tabela_categorias.head(15))

In [ ]:
# 4. Visualização
plt.figure(figsize=(12, 8))
sns.set_theme(style="whitegrid")

top_10 = tabela_categorias.head(10).reset_index()

# Plotando barras do 'Gargalo (Balanceado)'
ax = sns.barplot(x='Gargalo (Balanceado)', y='site_category_lv1', data=top_10, color='royalblue')
plt.title('Top 10 Categorias por Amostras Balanceadas (Amostras da Classe Minoritária)', fontsize=14)
plt.xlabel('Número máximo de amostras possíveis por classe (Balanceado)', fontsize=12)
plt.ylabel('Categoria (B2W)', fontsize=12)

# Anotando o número nos gráficos
for p in ax.patches:
    ax.annotate(f'{int(p.get_width()):,}', 
                (p.get_width(), p.get_y() + p.get_height() / 2.), 
                ha = 'left', va = 'center', 
                xytext = (5, 0), 
                textcoords = 'offset points')

plt.show()

As categorias com maior gargalo balanceado sao as candidatas mais seguras. Eletronicos (Celulares, Informatica, TV) e Beleza e Perfumaria atendem ao criterio com folga, viabilizando celulas de teste balanceadas sem rebaixar o tamanho do treino.

---
## Etapa 1 - Engenharia de dados

Esta e a etapa de maior cuidado metodologico. O objetivo e treinar em Ingles/Eletronicos e avaliar em quatro combinacoes idioma x dominio, isolando cada deslocamento. Decisoes centrais:

- **Filtragem hibrida por base.** No Ingles (Amazon, `amazon_polarity`, sem metadado de categoria) o dominio e atribuido por **palavras-chave** ineqivocas; no Portugues (B2W, com `site_category_lv1`) o dominio vem da **categoria oficial do produto** (*ground-truth*).
- **Auditoria do filtro Ingles** por um classificador *zero-shot* multilingue, para checar a precisao do filtro lexical.
- **Balanceamento desacoplado:** o treino usa todo o pool de S1; as celulas de teste vao a um N comum. Isso evita que o compartimento de teste mais escasso reduza o tamanho do treino.

### 1.1 Configuracao do ambiente

In [ ]:
# 3. Parâmetros centrais da Etapa 1
from pathlib import Path

DIR_SAIDA = Path("/content/data_processed")
DIR_SAIDA.mkdir(parents=True, exist_ok=True)

# Mapeamento estrela -> sentimento binário (Seção 4): 1,2->0(Neg) | 4,5->1(Pos) | 3 descartado
def estrela_para_sentimento(nota):
    try:
        n = float(nota)
    except (TypeError, ValueError):
        return None
    if n <= 2: return 0
    if n >= 4: return 1
    return None  # nota 3 (neutra) descartada

MAX_CANDIDATOS_EN = 20000      # candidatos coletados por domínio EN (antes do balanceamento)
MAX_LINHAS_STREAM_EN = 600000  # teto de linhas varridas no stream do Amazon
N_AUDITORIA = 100              # amostras por subconjunto para auditoria manual (Task 1.3)
LIMIAR_PRECISAO = 0.80         # critério de aceitação do filtro
print("Configurações carregadas.")

In [ ]:
### 1.2 Definicao de dominios - Ingles por palavra-chave, Portugues por categoria

As listas de palavras-chave do Ingles foram enxugadas para termos ineqivocos de eletronica de consumo e de beleza (termos ambiguos como *screen*, *camera* ou *processor* foram removidos por vazarem para outros dominios). No Portugues, o dominio vem da categoria do produto.

In [ ]:
### 1.2 Definicao de dominios - Ingles por palavra-chave, Portugues por categoria

As listas de palavras-chave do Ingles foram enxugadas para termos ineqivocos de eletronica de consumo e de beleza (termos ambiguos como *screen*, *camera* ou *processor* foram removidos por vazarem para outros dominios). No Portugues, o dominio vem da categoria do produto.

### 1.2 Definicao de dominios - Ingles por palavra-chave, Portugues por categoria

As listas de palavras-chave do Ingles foram enxugadas para termos ineqivocos de eletronica de consumo e de beleza (termos ambiguos como *screen*, *camera* ou *processor* foram removidos por vazarem para outros dominios). No Portugues, o dominio vem da categoria do produto.

In [ ]:
### 1.3 Carregamento dos pools (Ingles via streaming; Portugues via B2W)

In [ ]:
### 1.3 Carregamento dos pools (Ingles via streaming; Portugues via B2W)

### 1.3 Carregamento dos pools (Ingles via streaming; Portugues via B2W)

In [ ]:
### 1.4 Particionamento por dominio (S1-S4) e verificacao de disjuncao

In [ ]:
### 1.4 Particionamento por dominio (S1-S4) e verificacao de disjuncao

In [ ]:
### 1.5 Exploracao dos dados e diagnostico do gargalo de classe

### 1.4 Particionamento por dominio (S1-S4) e verificacao de disjuncao

In [ ]:
### 1.5 Exploracao dos dados e diagnostico do gargalo de classe

In [ ]:
### 1.6 Balanceamento desacoplado e split treino/validacao

O treino (S1) e balanceado no proprio maximo e dividido 80/20; as celulas de teste (S1_val, S2, S3, S4) sao balanceadas a um N comum, garantindo comparacao justa entre cenarios.

### 1.5 Exploracao dos dados e diagnostico do gargalo de classe

In [ ]:
### 1.6 Balanceamento desacoplado e split treino/validacao

O treino (S1) e balanceado no proprio maximo e dividido 80/20; as celulas de teste (S1_val, S2, S3, S4) sao balanceadas a um N comum, garantindo comparacao justa entre cenarios.

In [ ]:
### 1.7 Auditoria do filtro Ingles via classificacao *zero-shot*

So o lado Ingles (filtro lexical, sujeito a erro) e auditado por um classificador *zero-shot* multilingue. O lado Portugues vem da categoria do produto, *ground-truth* por construcao.

### 1.6 Balanceamento desacoplado e split treino/validacao

O treino (S1) e balanceado no proprio maximo e dividido 80/20; as celulas de teste (S1_val, S2, S3, S4) sao balanceadas a um N comum, garantindo comparacao justa entre cenarios.

In [ ]:
### 1.7 Auditoria do filtro Ingles via classificacao *zero-shot*

So o lado Ingles (filtro lexical, sujeito a erro) e auditado por um classificador *zero-shot* multilingue. O lado Portugues vem da categoria do produto, *ground-truth* por construcao.

In [ ]:
# Task 1.3 — (3/3) Estatísticas do filtro + Tabela de precisão final
import pandas as pd
from collections import Counter
import re
import matplotlib.pyplot as plt

# ---- EN: palavras-chave mais ativas (filtro lexical) ----
print("EN — palavras-chave mais frequentes (S1, S2):")
for S, nome in [(S1_raw,'S1'), (S2_raw,'S2')]:
    dom  = S['dominio'].iloc[0]
    kws  = KEYWORDS['en'][dom]
    cont = Counter()
    for t in S['texto'].str.lower():
        for kw in kws:
            if re.search(r"\b" + re.escape(kw) + r"\b", t):
                cont[kw] += 1
    top = ', '.join(f"{k} ({v})" for k, v in cont.most_common(5))
    print(f"  {nome} [en/{dom}]: {top}")

# ---- PT: categorias incluídas por domínio (site_category_lv1) ----
print("\nPT — categorias por domínio (site_category_lv1) — ground-truth, precisão 100%:")
for S, nome in [(S3_raw,'S3'), (S4_raw,'S4')]:
    dom  = S['dominio'].iloc[0]
    vc   = S['categoria'].value_counts()
    comp = ', '.join(f"{c} ({n:,})" for c, n in vc.items())
    print(f"  {nome} [pt/{dom}]: {comp}")

total_pt = len(df_pt)
atrib_pt = int(df_pt['dominio'].notna().sum())
print(f"\nSeletividade PT (por categoria): {atrib_pt:,}/{total_pt:,} "
      f"avaliações nos domínios escolhidos ({100*atrib_pt/total_pt:.1f}%).\n")

# ---- Tabela de precisão: EN por auditoria zero-shot; PT por categoria (ground-truth) ----
linhas = []
for nome, df_anot in resultados.items():
    prec   = df_anot['correto'].mean()
    status = "OK aprovado" if prec >= LIMIAR_PRECISAO else "FALHA refinar keywords"
    acord  = int(df_anot['correto'].sum())
    linhas.append([nome, str(len(df_anot)), str(acord), f"{prec:.1%}", status])
for nome in ['S3', 'S4']:
    linhas.append([nome, '—', '—', '100% (categoria)', 'OK ground-truth'])

tabela = pd.DataFrame(linhas, columns=[
    'Subconjunto', 'N amostrado', 'Acordos (filtro=LLM)', 'Precisão', 'Critério ≥ 80%'
])
print("Precisão da filtragem — EN por auditoria zero-shot; PT por categoria (ground-truth):")
display(tabela)

# ---- Gráfico: precisão dos subconjuntos EN auditados ----
aud = pd.DataFrame([[nome, df_anot['correto'].mean()] for nome, df_anot in resultados.items()],
                   columns=['Subconjunto', 'prec'])
fig, ax = plt.subplots(figsize=(5.5, 3.5))
cores = ['#2ecc71' if p >= LIMIAR_PRECISAO else '#e74c3c' for p in aud['prec']]
barras = ax.bar(aud['Subconjunto'], aud['prec'], color=cores, edgecolor='white')
ax.axhline(LIMIAR_PRECISAO, color='gray', linestyle='--', linewidth=1, label='Limiar 80%')
ax.set_ylim(0, 1.05); ax.set_ylabel('Precisão do filtro EN')
ax.set_title('Auditoria zero-shot — filtro lexical EN (S1, S2)')
for bar, p in zip(barras, aud['prec']):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.02, f"{p:.1%}",
            ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.legend(); plt.tight_layout(); plt.show()

falhas = aud[aud['prec'] < LIMIAR_PRECISAO]
if not falhas.empty:
    print("\nAVISO:  EN abaixo de 80%:", list(falhas['Subconjunto']), "-> refinar keywords EN.")
else:
    print("\nOK Filtro EN aprovado (≥ 80%). PT é ground-truth por categoria.")

### 1.7 Auditoria do filtro Ingles via classificacao *zero-shot*

So o lado Ingles (filtro lexical, sujeito a erro) e auditado por um classificador *zero-shot* multilingue. O lado Portugues vem da categoria do produto, *ground-truth* por construcao.

In [ ]:
# Gera o módulo src/data_pipeline.py a partir da lógica já validada nas seções 2.2–2.6
from pathlib import Path

DIR_SRC = Path('/content/src')
DIR_SRC.mkdir(parents=True, exist_ok=True)
(DIR_SRC / '__init__.py').write_text('', encoding='utf-8')

MODULO_DATA_PIPELINE = r'''# -*- coding: utf-8 -*-
"""
src/data_pipeline.py
====================
Módulo de preparação de dados da Etapa 1 do projeto
"Análise Arquitetural do Congelamento de Camadas em Transformers Multilíngues".

Atribuição de domínio HÍBRIDA:
  * EN (amazon_polarity, sem categoria) -> filtro lexical por palavra-chave (lista enxuta).
  * PT (B2W, com site_category_lv1)     -> categoria do produto (ground-truth).
"""
from __future__ import annotations

import os
import re
import itertools
from pathlib import Path
from typing import Dict, List, Optional

import pandas as pd

# ============================================================
# Domínios (Seção 5.2) — EN por keyword, PT por categoria
# ============================================================
KEYWORDS: Dict[str, Dict[str, List[str]]] = {
    "en": {
        "eletronicos": ["battery","usb","charger","charging","wifi","bluetooth",
                        "smartphone","laptop","tablet","touchscreen","headphone",
                        "headphones","earbuds","smartwatch","phone","phones","router","hdmi"],
        "beleza":      ["skin","scent","fragrance","perfume","cream",
                        "lotion","shampoo","conditioner","moisturizer","makeup",
                        "cosmetic","serum","sunscreen","soap","wrinkle"],
    },
}

CATEGORIAS_PT: Dict[str, str] = {
    "Celulares e Smartphones":  "eletronicos",
    "Informática e Acessórios": "eletronicos",
    "TV e Home Theater":        "eletronicos",
    "Beleza e Perfumaria":      "beleza",
}

URL_B2W = "https://raw.githubusercontent.com/americanas-tech/b2w-reviews01/main/B2W-Reviews01.csv"

# ============================================================
# Filtro lexical (EN)
# ============================================================
def _compilar_regex(palavras: List[str]) -> re.Pattern:
    partes = [re.escape(p.lower()) for p in palavras]
    return re.compile(r"\b(?:" + "|".join(partes) + r")\b", re.UNICODE)

REGEX = {lang: {dom: _compilar_regex(kws) for dom, kws in doms.items()}
         for lang, doms in KEYWORDS.items()}


def classificar_dominio(texto: str, lang: str = "en") -> Optional[str]:
    """EN: >=1 keyword do domínio E nenhuma do outro. Retorna 'eletronicos'|'beleza'|None."""
    if not isinstance(texto, str) or not texto.strip():
        return None
    t = texto.lower()
    tem_eletr  = bool(REGEX[lang]["eletronicos"].search(t))
    tem_beleza = bool(REGEX[lang]["beleza"].search(t))
    if tem_eletr and not tem_beleza:
        return "eletronicos"
    if tem_beleza and not tem_eletr:
        return "beleza"
    return None


def categoria_para_dominio(cat) -> Optional[str]:
    """PT: mapeia site_category_lv1 -> 'eletronicos'|'beleza'|None."""
    if not isinstance(cat, str):
        return None
    return CATEGORIAS_PT.get(cat.strip(), None)


# ============================================================
# Mapeamento estrela -> sentimento (Seção 4)
# ============================================================
def estrela_para_sentimento(nota) -> Optional[int]:
    """1-2 -> 0 (Negativo) | 4-5 -> 1 (Positivo) | 3 -> None (descartado)."""
    try:
        n = float(nota)
    except (TypeError, ValueError):
        return None
    if n <= 2: return 0
    if n >= 4: return 1
    return None


# ============================================================
# Carregamento (Task 1.1)
# ============================================================
def carregar_amazon_en_streaming(
    max_candidatos_por_dominio: int = 20000,
    max_linhas: int = 600000,
) -> pd.DataFrame:
    """Lê amazon_polarity em streaming e coleta candidatos por domínio (keyword)."""
    from datasets import load_dataset

    stream = None
    erros = []
    for repo in ["fancyzhx/amazon_polarity", "amazon_polarity"]:
        try:
            stream = load_dataset(repo, split="train", streaming=True)
            break
        except Exception as e:
            erros.append(f"{repo}: {e}")
    if stream is None:
        raise RuntimeError("Falha ao carregar Amazon EN:\n" + "\n".join(erros))

    coletado = {"eletronicos": [], "beleza": []}
    vistos = 0
    for ex in stream:
        vistos += 1
        texto = ((ex.get("title") or "") + ". " + (ex.get("content") or "")).strip()
        dom = classificar_dominio(texto, "en")
        if dom is not None and len(coletado[dom]) < max_candidatos_por_dominio:
            coletado[dom].append({
                "id": f"amz_{vistos}", "texto": texto,
                "label": int(ex["label"]), "idioma": "en", "dominio": dom,
            })
        if vistos >= max_linhas: break
        if (len(coletado["eletronicos"]) >= max_candidatos_por_dominio and
            len(coletado["beleza"]) >= max_candidatos_por_dominio): break
    return pd.DataFrame(coletado["eletronicos"] + coletado["beleza"])


def carregar_b2w(caminho: str = "/content/b2w.csv") -> pd.DataFrame:
    """Lê o B2W; baixa do repositório oficial se o arquivo não existir."""
    if os.path.exists(caminho):
        return pd.read_csv(caminho, low_memory=False)
    df = pd.read_csv(URL_B2W, low_memory=False)
    Path(caminho).parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(caminho, index=False)
    return df


def _col(df: pd.DataFrame, *nomes) -> Optional[str]:
    for n in nomes:
        if n in df.columns: return n
    return None


def _serie(df: pd.DataFrame, nome: Optional[str]) -> pd.Series:
    if nome is None:
        return pd.Series([""] * len(df))
    return df[nome].fillna("").astype(str)


def normalizar_b2w_pt(df_b2w: pd.DataFrame) -> pd.DataFrame:
    """Normaliza B2W em [id, texto, nota, label, idioma, categoria, dominio].

    O domínio vem de site_category_lv1 (categoria do produto), não de keyword.
    """
    b = pd.DataFrame()
    b["texto"] = (_serie(df_b2w, _col(df_b2w, "review_title")) + ". " +
                  _serie(df_b2w, _col(df_b2w, "review_text"))).str.strip()
    c_nota = _col(df_b2w, "overall_rating")
    b["nota"] = pd.to_numeric(df_b2w[c_nota], errors="coerce") if c_nota else None
    b["categoria"] = _serie(df_b2w, _col(df_b2w, "site_category_lv1"))
    b["label"] = b["nota"].apply(estrela_para_sentimento)
    b = b[b["texto"].str.len() > 0]
    b = b.dropna(subset=["label"]).copy()
    b["label"]   = b["label"].astype(int)
    b["idioma"]  = "pt"
    b["id"]      = ["pt_" + str(i) for i in range(len(b))]
    b["dominio"] = b["categoria"].apply(categoria_para_dominio)
    return b


# ============================================================
# Particionamento (Task 1.2)
# ============================================================
def construir_particoes(df_en: pd.DataFrame, df_pt: pd.DataFrame) -> Dict[str, pd.DataFrame]:
    """Aplica a partição S1..S4 sobre os pools rotulados e VERIFICA a disjunção de IDs."""
    S1 = df_en[df_en["dominio"] == "eletronicos"].copy()  # EN Eletrônicos (keyword)
    S2 = df_en[df_en["dominio"] == "beleza"].copy()        # EN Beleza (keyword)
    S3 = df_pt[df_pt["dominio"] == "eletronicos"].copy()  # PT Eletrônicos (categoria)
    S4 = df_pt[df_pt["dominio"] == "beleza"].copy()        # PT Beleza (categoria)

    ids = {"S1": set(S1["id"]), "S2": set(S2["id"]),
           "S3": set(S3["id"]), "S4": set(S4["id"])}
    for a, b in itertools.combinations(ids, 2):
        if ids[a] & ids[b]:
            raise AssertionError(f"Interseção de IDs entre {a} e {b}")
    return {"S1": S1, "S2": S2, "S3": S3, "S4": S4}


# ============================================================
# Balanceamento e split (Seção 5.4) — treino DESACOPLADO dos testes
# ============================================================
def _min_classe(S: pd.DataFrame) -> int:
    return min(int((S["label"] == 0).sum()), int((S["label"] == 1).sum()))


def balancear(S: pd.DataFrame, n: int, seed: int = 42) -> pd.DataFrame:
    pos = S[S["label"] == 1].sample(n=n, random_state=seed)
    neg = S[S["label"] == 0].sample(n=n, random_state=seed)
    return pd.concat([pos, neg]).sample(frac=1, random_state=seed).reset_index(drop=True)


def balancear_e_dividir(
    particoes: Dict[str, pd.DataFrame], seed: int = 42, test_size: float = 0.2,
) -> Dict[str, pd.DataFrame]:
    """Treino (S1) balanceado no próprio máximo e dividido 80/20; testes a um N comum."""
    from sklearn.model_selection import train_test_split

    cols = ["id", "idioma", "dominio", "label", "texto"]

    N_s1 = _min_classe(particoes["S1"])
    if N_s1 <= 0:
        raise ValueError("S1 (treino) ficou sem exemplos de uma das classes.")
    S1_full = balancear(particoes["S1"], N_s1, seed=seed)[cols]
    S1_train, S1_val_full = train_test_split(
        S1_full, test_size=test_size, stratify=S1_full["label"], random_state=seed)

    N_test = min(_min_classe(S1_val_full),
                 _min_classe(particoes["S2"]),
                 _min_classe(particoes["S3"]),
                 _min_classe(particoes["S4"]))
    if N_test <= 0:
        raise ValueError("Algum compartimento de teste ficou sem exemplos de uma das classes.")

    return {
        "S1_train": S1_train,
        "S1_val": balancear(S1_val_full, N_test, seed=seed)[cols],
        "S2": balancear(particoes["S2"], N_test, seed=seed)[cols],
        "S3": balancear(particoes["S3"], N_test, seed=seed)[cols],
        "S4": balancear(particoes["S4"], N_test, seed=seed)[cols],
        "N_s1": N_s1, "N_test": N_test,
    }


# ============================================================
# Orquestrador
# ============================================================
def preparar_etapa1(
    dir_saida: str = "/content/data_processed",
    caminho_b2w: str = "/content/b2w.csv",
    max_candidatos_en: int = 20000,
    max_linhas_amazon: int = 600000,
    seed: int = 42,
) -> Dict[str, pd.DataFrame]:
    """Executa Tasks 1.1 e 1.2 ponta a ponta e salva os 5 CSVs em `dir_saida`."""
    dir_saida = Path(dir_saida); dir_saida.mkdir(parents=True, exist_ok=True)

    df_en = carregar_amazon_en_streaming(max_candidatos_en, max_linhas_amazon)
    df_pt = normalizar_b2w_pt(carregar_b2w(caminho_b2w))
    particoes = construir_particoes(df_en, df_pt)
    artefatos = balancear_e_dividir(particoes, seed=seed)

    artefatos["S1_train"].to_csv(dir_saida/"S1_train_en_eletronicos.csv", index=False)
    artefatos["S1_val"  ].to_csv(dir_saida/"S1_val_en_eletronicos.csv",   index=False)
    artefatos["S2"      ].to_csv(dir_saida/"S2_en_beleza.csv",        index=False)
    artefatos["S3"      ].to_csv(dir_saida/"S3_pt_eletronicos.csv",   index=False)
    artefatos["S4"      ].to_csv(dir_saida/"S4_pt_beleza.csv",        index=False)
    return artefatos


# ============================================================
# Auto-teste minimalista
# ============================================================
if __name__ == "__main__":
    assert classificar_dominio("the battery life is amazing", "en") == "eletronicos"
    assert classificar_dominio("this moisturizer cream is great for my skin", "en") == "beleza"
    assert classificar_dominio("the laptop smells like perfume", "en") is None
    assert categoria_para_dominio("Celulares e Smartphones") == "eletronicos"
    assert categoria_para_dominio("Beleza e Perfumaria") == "beleza"
    assert categoria_para_dominio("Livros") is None
    assert estrela_para_sentimento(1) == 0
    assert estrela_para_sentimento(5) == 1
    assert estrela_para_sentimento(3) is None
    print("data_pipeline OK")
'''

destino = DIR_SRC / 'data_pipeline.py'
destino.write_text(MODULO_DATA_PIPELINE, encoding='utf-8')
print(f'Módulo gerado: {destino} ({destino.stat().st_size:,} bytes)')

In [ ]:
### 1.8 Consolidacao modular - `src/data_pipeline.py`

A logica validada e encapsulada em um modulo reutilizavel, exercitado por um *smoke test*.

In [ ]:
---
## Etapa 2 - Arquitetura e logica de congelamento

Carregamos o **XLM-RoBERTa-base** (encoder de 12 camadas, ~278M de parametros) com a *head* padrao de classificacao do Hugging Face, e implementamos o congelamento C1-C4 via `requires_grad` por nome de parametro. A logica e idempotente (pode trocar de config sem recarregar o modelo) e mantem a *head* sempre treinavel.

### 2.0 Setup e carga dos dados

### 1.8 Consolidacao modular - `src/data_pipeline.py`

A logica validada e encapsulada em um modulo reutilizavel, exercitado por um *smoke test*.

In [ ]:
---
## Etapa 2 - Arquitetura e logica de congelamento

Carregamos o **XLM-RoBERTa-base** (encoder de 12 camadas, ~278M de parametros) com a *head* padrao de classificacao do Hugging Face, e implementamos o congelamento C1-C4 via `requires_grad` por nome de parametro. A logica e idempotente (pode trocar de config sem recarregar o modelo) e mantem a *head* sempre treinavel.

### 2.0 Setup e carga dos dados

In [ ]:
### 2.1 Carregamento do XLM-RoBERTa + *head* de classificacao

### 1.9 Resumo da Etapa 1

Cinco artefatos sao produzidos: `S1_train` (treino), `S1_val` (T1), `S2` (T2), `S3` (T3) e `S4` (T4). O treino fica com milhares de exemplos (em vez de algumas centenas, como ocorria na regra acoplada antiga), e cada celula de teste fica balanceada 50/50 no mesmo N.

---
## Etapa 2 - Arquitetura e logica de congelamento

Carregamos o **XLM-RoBERTa-base** (encoder de 12 camadas, ~278M de parametros) com a *head* padrao de classificacao do Hugging Face, e implementamos o congelamento C1-C4 via `requires_grad` por nome de parametro. A logica e idempotente (pode trocar de config sem recarregar o modelo) e mantem a *head* sempre treinavel.

### 2.0 Setup e carga dos dados

In [ ]:
### 2.1 Carregamento do XLM-RoBERTa + *head* de classificacao

In [ ]:
# 2.2 (2/2) — VERIFY: sanidade da lógica de congelamento
import re

total = M.contar_parametros(model_xlmr)["total"]
head  = sum(p.numel() for n, p in model_xlmr.named_parameters() if n.startswith("classifier"))
treinaveis = {cfg: M.freeze_layers(model_xlmr, cfg)["treinavel"] for cfg in ["C1", "C2", "C3", "C4"]}

assert treinaveis["C1"] == total,  f"C1 deveria treinar tudo: {treinaveis['C1']} vs {total}"
assert treinaveis["C4"] == head,   f"C4 deveria treinar só a head ({head:,}): {treinaveis['C4']:,}"
assert treinaveis["C2"] < treinaveis["C3"] < treinaveis["C1"], f"ordem C2<C3<C1 falhou: {treinaveis}"
assert tokenizer.is_fast, "tokenizer não é fast (AutoTokenizer use_fast)"

# Varredura por prefixo na C2: embeddings+0–5 congelados; 6–11 treináveis; head treinável.
M.freeze_layers(model_xlmr, "C2")
pat = re.compile(r"^roberta\.encoder\.layer\.(\d+)\.")
for n, p in model_xlmr.named_parameters():
    if n.startswith("classifier"):
        assert p.requires_grad, f"C2: head congelada? {n}"
    elif n.startswith("roberta.embeddings"):
        assert not p.requires_grad, f"C2: embeddings deveria estar congelado: {n}"
    else:
        m = pat.match(n)
        if m:
            i = int(m.group(1))
            esperado_treinavel = i >= 6
            assert p.requires_grad == esperado_treinavel, f"C2: camada {i} requires_grad errado: {n}"

M.freeze_layers(model_xlmr, "C1")  # restaura tudo treinável (estado limpo p/ a Fase 2.4)
print("OK Fase 2.2 — freeze_layers C1-C4 validado | tokenizer.is_fast =", tokenizer.is_fast)
print("head (classifier) =", f"{head:,}", "params | total =", f"{total:,}")
print("Proximo: Fase 2.3 (tests/test_model.py + atualizar §6) e 2.4 (smoke test fwd+bwd).")

In [ ]:
### 2.3 Testes formais da logica de congelamento

In [ ]:
# 2.3 (1/2) — Gera tests/test_model.py e espelha no Drive
from pathlib import Path
import shutil, sys

DIR_TESTS = Path('/content/tests'); DIR_TESTS.mkdir(parents=True, exist_ok=True)
(DIR_TESTS / '__init__.py').write_text('', encoding='utf-8')

MODULO_TESTS = r'''# -*- coding: utf-8 -*-
"""
tests/test_model.py
===================
Testes de verificação da Etapa 2 (Fase 2.3) — congelamento seletivo C1-C4.
Confere requires_grad por prefixo, head sempre treinável e a contagem real de
parâmetros treináveis por config. Roda standalone (python tests/test_model.py)
ou importado no notebook chamando rodar_todos(model) — sem recarregar o modelo.
"""
from __future__ import annotations

import re
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))
from src import model as M

_PAD = re.compile(r"^roberta\.encoder\.layer\.(\d+)\.")

# Especificação esperada por config: (congela_embeddings, camadas congeladas).
_ESPEC = {
    "C1": (False, set()),
    "C2": (True, set(range(0, 6))),
    "C3": (False, set(range(6, 12))),
    "C4": (True, set(range(0, 12))),
}

# Contagens reais confirmadas no run da Fase 2.2 (regressão).
_ESPERADO_TREINAVEL = {
    "C1": 278_045_186,
    "C2": 43_119_362,
    "C3": 235_517_954,
    "C4": 592_130,
}


def _head(model):
    return sum(p.numel() for n, p in model.named_parameters() if n.startswith("classifier"))


def teste_c1_treina_tudo(model):
    c = M.freeze_layers(model, "C1")
    assert c["treinavel"] == c["total"], c


def teste_c4_so_head(model):
    c = M.freeze_layers(model, "C4")
    assert c["treinavel"] == _head(model), (c, _head(model))


def teste_head_sempre_treinavel(model):
    for cfg in M.CONFIGS:
        M.freeze_layers(model, cfg)
        congeladas = [n for n, p in model.named_parameters()
                      if n.startswith("classifier") and not p.requires_grad]
        assert not congeladas, (cfg, congeladas)


def teste_prefixos_por_config(model):
    for cfg, (emb_cong, cam_cong) in _ESPEC.items():
        M.freeze_layers(model, cfg)
        for n, p in model.named_parameters():
            if n.startswith("classifier"):
                assert p.requires_grad, (cfg, n)
            elif n.startswith("roberta.embeddings"):
                assert p.requires_grad == (not emb_cong), (cfg, n)
            else:
                m = _PAD.match(n)
                if m is not None:
                    i = int(m.group(1))
                    assert p.requires_grad == (i not in cam_cong), (cfg, n)


def teste_contagem_real(model):
    for cfg, val in _ESPERADO_TREINAVEL.items():
        c = M.freeze_layers(model, cfg)
        assert c["treinavel"] == val, (cfg, c["treinavel"], val)


def teste_idempotente(model):
    a = M.freeze_layers(model, "C2")["treinavel"]
    b = M.freeze_layers(model, "C2")["treinavel"]
    M.freeze_layers(model, "C3")
    c = M.freeze_layers(model, "C2")["treinavel"]
    assert a == b == c, (a, b, c)


def teste_config_invalida(model):
    try:
        M.freeze_layers(model, "C9")
    except ValueError:
        return
    raise AssertionError("freeze_layers deveria levantar ValueError para config inválida")


TESTES = [
    teste_c1_treina_tudo,
    teste_c4_so_head,
    teste_head_sempre_treinavel,
    teste_prefixos_por_config,
    teste_contagem_real,
    teste_idempotente,
    teste_config_invalida,
]


def rodar_todos(model):
    falhas = []
    for t in TESTES:
        try:
            t(model)
            print("  PASS", t.__name__)
        except AssertionError as e:
            falhas.append(t.__name__)
            print("  FAIL", t.__name__, "->", e)
    M.freeze_layers(model, "C1")  # restaura estado limpo
    if falhas:
        raise AssertionError("%d teste(s) falharam: %s" % (len(falhas), falhas))
    print("OK %d testes passaram." % len(TESTES))
    return True


if __name__ == "__main__":
    modelo, _ = M.carregar_modelo(seed=42)
    rodar_todos(modelo)
'''

(DIR_TESTS / 'test_model.py').write_text(MODULO_TESTS, encoding='utf-8')
shutil.copytree('/content/tests', DIR_PROJ / 'tests', dirs_exist_ok=True)  # persiste no Drive
if '/content' not in sys.path: sys.path.insert(0, '/content')
print('OK tests/test_model.py gerado (%d bytes) e espelhado em %s' % ((DIR_TESTS/'test_model.py').stat().st_size, DIR_PROJ/'tests'))

In [ ]:
### 2.2 Logica de congelamento C1-C4 e tabela real de parametros treinaveis

A tabela abaixo confirma a hierarquia esperada de parametros treinaveis: C4 (so *head*) < C2 (*Freeze Lower*) < C3 (*Freeze Upper*) < C1 (tudo). Como os embeddings concentram a maior parte dos parametros, congela-los (C2 e C4) reduz drasticamente o numero de pesos treinaveis.

### 2.1 Carregamento do XLM-RoBERTa + *head* de classificacao

In [ ]:
# 2.4 — Smoke test: 1 batch real de S1_train; forward+backward por config.
#       Prova que o gradiente flui SO nos parametros treinaveis.
import torch

B = 8
amostra = dados["S1_train"].sample(B, random_state=SEEDS[0])
enc = tokenizer(list(amostra["texto"]), truncation=True, padding="max_length",
                max_length=128, return_tensors="pt").to(device)
labels = torch.tensor(amostra["label"].tolist(), device=device)

def smoke(model, config):
    M.freeze_layers(model, config)
    model.train(); model.zero_grad(set_to_none=True)
    out = model(**enc, labels=labels)
    out.loss.backward()
    congelados_com_grad, treina_com_grad, head_ok = [], 0, True
    for n, p in model.named_parameters():
        grad_nz = (p.grad is not None) and bool(torch.any(p.grad != 0))
        if p.requires_grad:
            treina_com_grad += int(grad_nz)
            if n.startswith("classifier") and not grad_nz:
                head_ok = False
        elif grad_nz:
            congelados_com_grad.append(n)
    return out.loss.item(), treina_com_grad, congelados_com_grad, head_ok

print(f"{'config':>6} | {'loss':>8} | {'tensores treinaveis c/ grad':>27} | {'congelados c/ grad':>18}")
print('-' * 70)
for cfg in ["C1", "C2", "C3", "C4"]:
    loss, treina, congel, head_ok = smoke(model_xlmr, cfg)
    assert torch.isfinite(torch.tensor(loss)), f"{cfg}: loss nao finita ({loss})"
    assert not congel, f"{cfg}: {len(congel)} tensores CONGELADOS receberam gradiente! ex: {congel[:2]}"
    assert head_ok, f"{cfg}: a head (classifier) NAO recebeu gradiente"
    print(f"{cfg:>6} | {loss:8.4f} | {treina:>27,} | {len(congel):>18}")

M.freeze_layers(model_xlmr, "C1"); model_xlmr.zero_grad(set_to_none=True)
print("\nOK Fase 2.4 — gradiente flui SO nos treinaveis (loss finita; 0 congelados com grad; head sempre com grad).")
print("ETAPA 2 COMPLETA: src/model.py (carregar_modelo + freeze_layers) testado e pronto p/ a Etapa 3.")

In [ ]:
### 2.3 Testes formais da logica de congelamento

In [ ]:
# 3.0 (1/7) — Bibliotecas
!pip install -q transformers datasets accelerate evaluate scikit-learn
print("Bibliotecas instaladas.")

### 2.2 Logica de congelamento C1-C4 e tabela real de parametros treinaveis

A tabela abaixo confirma a hierarquia esperada de parametros treinaveis: C4 (so *head*) < C2 (*Freeze Lower*) < C3 (*Freeze Upper*) < C1 (tudo). Como os embeddings concentram a maior parte dos parametros, congela-los (C2 e C4) reduz drasticamente o numero de pesos treinaveis.

In [ ]:
### 2.4 *Smoke test* com dados reais (forward + backward)

Prova que o gradiente flui **apenas** nos parametros treinaveis: *loss* finita, zero parametros congelados com gradiente, e *head* sempre com gradiente.

In [ ]:
# 3.0 (4/7) — Monta o Drive: dados de entrada + pasta de resultados
from google.colab import drive
drive.mount('/content/drive')

DIR_PROJ = Path('/content/drive/MyDrive/TrabalhoRNP')
DIR_DATA = DIR_PROJ / 'data_processed'
DIR_RES  = DIR_PROJ / 'resultados'
(DIR_RES / 'runs').mkdir(parents=True, exist_ok=True)

assert DIR_DATA.exists(), f"Sem {DIR_DATA} — confira a persistência da Etapa 1/2 no Drive."
print("Dados  :", DIR_DATA)
print("Saídas :", DIR_RES, "(runs/ + results.csv chegam na 3.2/3.3)")

### 2.3 Testes formais da logica de congelamento

In [ ]:
%%writefile /content/Trabalho-de-Redes-Neurais-Profundas/src/train.py
import numpy as np
import evaluate
from transformers import TrainingArguments, EarlyStoppingCallback

# Carrega as métricas oficiais
f1_metric = evaluate.load("f1")
acc_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    
    # F1-macro e Accuracy
    f1_macro = f1_metric.compute(predictions=predictions, references=labels, average="macro")["f1"]
    acc = acc_metric.compute(predictions=predictions, references=labels)["accuracy"]
    
    # F1 por classe (0 = Negativo, 1 = Positivo) para ter precisão granular
    f1_classes = f1_metric.compute(predictions=predictions, references=labels, average=None)["f1"]
    
    return {
        "f1_macro": f1_macro,
        "accuracy": acc,
        "f1_negativo": f1_classes[0],
        "f1_positivo": f1_classes[1]
    }

def criar_training_args(out_dir, seed):
    return TrainingArguments(
        output_dir=out_dir,
        num_train_epochs=3,
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        weight_decay=0.01,
        warmup_ratio=0.10,
        fp16=True,                     # Mix precision (ideal para T4)
        eval_strategy="epoch",         # Avalia validação a cada época concluída
        save_strategy="epoch",         # Salva checkpoints a cada época
        load_best_model_at_end=True,   # Puxa o melhor modelo quando o early stopping ativar
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        seed=seed,
        logging_strategy="epoch",      # Log de loss
        report_to="none"               # Desabilita integrações com wandb para não pedir login
    )

def get_early_stopping():
    return EarlyStoppingCallback(early_stopping_patience=1)


In [ ]:
import json
import shutil
import gc
import torch
import pandas as pd
from pathlib import Path
from transformers import Trainer, DataCollatorWithPadding

import sys
if "src.train" in sys.modules:
    del sys.modules["src.train"]
from src import train as T
from src import model as M

RESULTS_CSV = DIR_RES / "results.csv"

def treinar_run(config, seed):
    print(f"\n{'='*60}\n Iniciando run: Config = {config} | Seed = {seed}\n{'='*60}")
    
    # 1. Carregar modelo e tokenizer (seed fixada)
    model, tok = M.carregar_modelo(seed)
    params = M.freeze_layers(model, config)
    print(f"[{config}-{seed}] Modelo carregado. Parâmetros treináveis: {params['treinavel']:,} / {params['total']:,}")
    
    # 2. Configurar o Trainer e o DataCollator (Dynamic Padding)
    out_dir = f"/content/tmp_trainer_{config}_{seed}"
    args = T.criar_training_args(out_dir, seed)
    data_collator = DataCollatorWithPadding(tokenizer=tok)
    
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=ds_train,
        eval_dataset=ds_val,
        compute_metrics=T.compute_metrics,
        callbacks=[T.get_early_stopping()],
        data_collator=data_collator
    )
    
    # 3. Dispara o Treino
    print(f"[{config}-{seed}] Treinamento iniciado...")
    trainer.train()
    
    # 4. Salvar histórico de treino (curvas de loss e épocas) na pasta `runs/`
    best_eval_loss = trainer.state.best_metric
    hist = {
        "config": config,
        "seed": seed,
        "best_eval_loss": best_eval_loss,
        "log_history": trainer.state.log_history
    }
    json_path = DIR_RES / f"runs/run_{config}_{seed}.json"
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(hist, f, indent=2)
    print(f"[{config}-{seed}] Curvas de perda salvas em {json_path.name}")
    
    # 5. Avaliação inline em T1-T4 (D4: não guardaremos o modelo, só os resultados)
    print(f"[{config}-{seed}] Avaliando o modelo treinado nos testes T1 a T4...")
    linhas_resultado = []
    
    for teste_nome, ds_teste in ds_testes.items():
        metrics = trainer.evaluate(eval_dataset=ds_teste, metric_key_prefix="eval")
        
        linha = {
            "config": config,
            "seed": seed,
            "teste": teste_nome,
            "f1_macro": metrics.get("eval_f1_macro"),
            "accuracy": metrics.get("eval_accuracy"),
            "f1_negativo": metrics.get("eval_f1_negativo"),
            "f1_positivo": metrics.get("eval_f1_positivo")
        }
        linhas_resultado.append(linha)
    
    # Append seguro das 4 linhas no CSV de resultados (cria ou atualiza arquivo)
    df_novo = pd.DataFrame(linhas_resultado)
    if RESULTS_CSV.exists():
        df_novo.to_csv(RESULTS_CSV, mode="a", header=False, index=False)
    else:
        df_novo.to_csv(RESULTS_CSV, mode="w", header=True, index=False)
    print(f"[{config}-{seed}] OK 4 resultados adicionados em results.csv")
    
    # 6. Apagar checkpoint local para não estourar o disco (D4)
    if Path(out_dir).exists():
        shutil.rmtree(out_dir)
        print(f"[{config}-{seed}]  Checkpoints temporários descartados.\n")
        
    # Limpa GPU para garantir que o Colab não sofra OOM nos próximos ciclos
    del model, trainer
    gc.collect()
    torch.cuda.empty_cache()

# ==========================================
# FASE 3.3 - LOOP DOS 12 TREINOS (RESUMÍVEL)
# ==========================================
def executar_pipeline():
    runs_feitos = set()
    if RESULTS_CSV.exists():
        df_res = pd.read_csv(RESULTS_CSV)
        for _, row in df_res.iterrows():
            runs_feitos.add((row["config"], row["seed"]))
    
    configs = ["C1", "C2", "C3", "C4"]
    for config in configs:
        for seed in SEEDS:
            if (config, seed) in runs_feitos:
                print(f" Pulando run já processado: {config} / seed={seed}")
                continue
            
            treinar_run(config, seed)
            
    print("\n Pipeline da Etapa 3 concluído com sucesso!")

# Chamada principal: inicia os treinamentos
executar_pipeline()


### 2.4 *Smoke test* com dados reais (forward + backward)

Prova que o gradiente flui **apenas** nos parametros treinaveis: *loss* finita, zero parametros congelados com gradiente, e *head* sempre com gradiente.

In [ ]:
---
## Etapa 4 - Analise estatistica consolidada (multi-seed)

Esta etapa e **auto-contida**: a base consolidada esta embutida na proxima celula, e toda a analise roda **sem GPU e sem Drive**. A base reune as execucoes independentes dos integrantes do grupo, com **6 a 8 seeds por configuracao** nos cenarios solidos (T1-T4). O numero maior de seeds em uma configuracao apenas aumenta a precisao da estimativa; os testes usados (Welch e Mann-Whitney) toleram N desigual.

Os resultados a seguir cobrem os deslocamentos que o desenho mede de forma limpa: **Domain Shift** (T2) e **Language Shift proximo** (T3, Ingles para Portugues). O *Language Shift* distante (Japones/Mandarim) e tratado na Etapa 9.

---
## Etapa 3 - Treino dos modelos e extracao de metricas

Cada modelo e treinado com **AdamW**, *learning rate* 2e-5, ate 3 epocas, *warmup* de 10%, *weight decay* 0.01 e *early stopping* (paciencia 1) sobre a *loss* de validacao - um split interno de 10% do treino, deixando S1_val intocado como T1. Cada modelo treinado e avaliado nas celulas de teste, gerando as metricas de F1-macro e acuracia.

O *loop* e **resumivel**: relê o CSV de resultados e pula execucoes ja feitas. A base final deste estudo agrega esse mesmo procedimento aplicado por varios integrantes, cada um com suas seeds independentes.

### 3.0 Setup, carga dos dados e split de validacao

In [ ]:
### 4.2 Delta-shift: queda de F1-macro vs. baseline T1 (em pontos percentuais)

Convencao: `Delta = F1(T1) - F1(Tx)`. Valor **> 0 = perda** sob deslocamento; **< 0 = ganho** *zero-shot*. As colunas de **lingua proxima** sao negativas - o modelo vai **igual ou melhor** em Portugues, apesar de nunca ter treinado nesse idioma.

In [ ]:
# Tabela 4.2 - Delta-shift por configuracao (>0 = perda; <0 = ganho zero-shot)
deltas = pd.DataFrame({
    "Delta Dominio (T1-T2) pp":       (media["T1"] - media["T2"]) * 100,
    "Delta Lingua prox. (T1-T3) pp":  (media["T1"] - media["T3"]) * 100,
    "Delta Combinado (T1-T4) pp":     (media["T1"] - media["T4"]) * 100,
}).round(2)
deltas

In [ ]:
> **Da execucao individual a base consolidada.** Este *loop* produz um `results.csv` por execucao. Para o estudo final, o procedimento foi repetido por tres integrantes do grupo com **seeds independentes**, e os CSVs foram combinados em uma unica base (6 a 8 seeds por configuracao). E essa base consolidada que a Etapa 4 analisa - uma **replicacao independente** que estressa as conclusoes muito alem das seeds iniciais.

In [ ]:
---
## Etapa 4 - Analise estatistica consolidada (multi-seed)

Esta etapa e **auto-contida**: a base consolidada esta embutida na proxima celula, e toda a analise roda **sem GPU e sem Drive**. A base reune as execucoes independentes dos integrantes do grupo, com **6 a 8 seeds por configuracao** nos cenarios solidos (T1-T4). O numero maior de seeds em uma configuracao apenas aumenta a precisao da estimativa; os testes usados (Welch e Mann-Whitney) toleram N desigual.

Os resultados a seguir cobrem os deslocamentos que o desenho mede de forma limpa: **Domain Shift** (T2) e **Language Shift proximo** (T3, Ingles para Portugues). O *Language Shift* distante (Japones/Mandarim) e tratado na Etapa 9.

In [ ]:
# Etapa 4 - base consolidada embutida (self-contained, sem GPU/Drive).
# Reune as execucoes independentes dos integrantes do grupo (seeds distintas por
# pessoa): 6 a 8 seeds por configuracao, nos cenarios solidos T1-T4. E uma
# replicacao independente que estressa a robustez das conclusoes.
%matplotlib inline
import io
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

BASE_CONSOLIDADA = """\
config,seed,teste,f1_macro,accuracy,f1_negativo,f1_positivo
C1,13,T1,0.9466396439709718,0.946641791044776,0.9469781238413052,0.9463011641006384
C1,13,T2,0.9207807700952426,0.9208955223880596,0.9177657098525988,0.9237958303378864
C1,13,T3,0.957450150996813,0.9574626865671642,0.9567198177676538,0.958180484225972
C1,13,T4,0.9499618499513164,0.95,0.9485801995395242,0.9513435003631082
C1,888,T1,0.9350277322014552,0.9350746268656716,0.9367732558139537,0.933282208588957
C1,888,T2,0.9123016988656452,0.9123134328358208,0.9133161195130948,0.9112872782181956
C1,888,T3,0.95932708406576,0.9593283582089552,0.9590994371482175,0.9595547309833024
C1,888,T4,0.9582066276803118,0.9582089552238806,0.9578947368421052,0.9585185185185184
C1,1234,T1,0.941037387732368,0.941044776119403,0.940377358490566,0.9416974169741698
C1,1234,T2,0.9086376741615294,0.908955223880597,0.9032513877874704,0.9140239605355884
C1,1234,T3,0.9537032651234794,0.953731343283582,0.9525631216526396,0.954843408594319
C1,1234,T4,0.9480796133037838,0.948134328358209,0.9463941380640184,0.949765088543549
C1,2718,T1,0.9436565203014324,0.9436567164179104,0.9435514018691588,0.9437616387337058
C1,2718,T2,0.9177616437851738,0.917910447761194,0.9142634450506624,0.9212598425196852
C1,2718,T3,0.9525933998190148,0.9526119402985076,0.951655881233346,0.9535309184046836
C1,2718,T4,0.9499618499513164,0.95,0.9485801995395242,0.9513435003631082
C1,4242,T1,0.9455114332966252,0.9455223880597016,0.9462840323767476,0.9447388342165026
C1,4242,T2,0.9242144689851712,0.9242537313432836,0.9224894998090876,0.925939438161255
C1,4242,T3,0.9552028991079023,0.9552238805970148,0.954233409610984,0.956172388604821
C1,4242,T4,0.9507023411371238,0.9507462686567164,0.9492307692307692,0.9521739130434784
C1,5678,T1,0.9380149629048368,0.9380597014925374,0.9396802325581396,0.9363496932515336
C1,5678,T2,0.926491298931718,0.9264925373134328,0.9267930137495356,0.9261895841139004
C1,5678,T3,0.9567109935250296,0.9567164179104478,0.9562264150943396,0.9571955719557196
C1,5678,T4,0.9540926594630412,0.9541044776119404,0.9533560864618884,0.954829232464194
C1,9001,T1,0.9373084052185072,0.9373134328358208,0.9367469879518072,0.9378698224852072
C1,9001,T2,0.9116485655743428,0.9119402985074628,0.906571654790182,0.916725476358504
C1,9001,T3,0.9514492753623188,0.9514925373134328,0.95,0.9528985507246376
C1,9001,T4,0.9416998411973286,0.9417910447761194,0.9393939393939394,0.9440057430007178
C1,91011,T1,0.942163977792861,0.9421641791044776,0.9422718808193667,0.9420560747663552
C1,91011,T2,0.9174421277372176,0.9175373134328358,0.9146388567014292,0.9202453987730062
C1,91011,T3,0.9552118080391464,0.9552238805970148,0.9544764795144158,0.9559471365638766
C1,91011,T4,0.9473511078378553,0.9473880597014924,0.9459563050977386,0.9487459105779716
C2,1234,T1,0.925353013398577,0.9253731343283582,0.9265785609397944,0.9241274658573596
C2,1234,T2,0.9167420688406176,0.9167910447761194,0.9147227533460804,0.9187613843351548
C2,1234,T3,0.9514682119574304,0.9514925373134328,0.950381679389313,0.9525547445255474
C2,1234,T4,0.9495953627733752,0.9496268656716418,0.9483352468427096,0.9508554787040407
C2,2718,T1,0.939922353526624,0.9399253731343284,0.9394964299135664,0.9403482771396814
C2,2718,T2,0.9184987157660848,0.9186567164179104,0.9149102263856362,0.9220872051465332
C2,2718,T3,0.9540791016867056,0.9541044776119404,0.9529996178830722,0.955158585490339
C2,2718,T4,0.9480892215459552,0.948134328358209,0.946559015763168,0.9496194273287424
C2,4242,T1,0.9410383401173218,0.941044776119403,0.9416543574593796,0.940422322775264
C2,4242,T2,0.923434024420003,0.9235074626865672,0.9210627647285328,0.925805284111473
C2,4242,T3,0.9593188370133964,0.9593283582089552,0.9586964759378552,0.959941198088938
C2,4242,T4,0.9522148548503928,0.9522388059701492,0.9511450381679388,0.9532846715328468
C2,5678,T1,0.938801846192576,0.9388059701492536,0.9393042190969652,0.9382994732881866
C2,5678,T2,0.9227038439497116,0.9227611940298508,0.9205983889528192,0.9248092989466036
C2,5678,T3,0.9522148548503928,0.9522388059701492,0.9511450381679388,0.9532846715328468
C2,5678,T4,0.9492229359418806,0.9492537313432836,0.9479724560061208,0.9504734158776402
C2,9001,T1,0.9384324157925448,0.9384328358208956,0.9382716049382716,0.938593226646818
C2,9001,T2,0.9169375104707655,0.9171641791044776,0.9125984251968504,0.9212765957446808
C2,9001,T3,0.95108109143731,0.9511194029850746,0.9497120921305182,0.9524500907441016
C2,9001,T4,0.9458274522104309,0.9458955223880596,0.943907156673114,0.9477477477477476
C2,91011,T1,0.9402982082222512,0.9402985074626866,0.9401645474943904,0.9404318689501115
C2,91011,T2,0.918880484853557,0.9190298507462686,0.9153996101364522,0.922361359570662
C2,91011,T3,0.9533272709870196,0.9533582089552238,0.9521256223669092,0.9545289196071298
C2,91011,T4,0.948084533631124,0.948134328358209,0.9464767038891028,0.9496923633731452
C3,1234,T1,0.9231322307191496,0.923134328358209,0.9227306826706676,0.9235337787676318
C3,1234,T2,0.880539019257934,0.8809701492537313,0.8733624454148472,0.8877155931010208
C3,1234,T3,0.920708952620194,0.9208955223880596,0.9168627450980392,0.9245551601423488
C3,1234,T4,0.9164744398285156,0.9167910447761194,0.9113320079522864,0.9216168717047452
C3,2718,T1,0.9227521489414764,0.9227611940298508,0.9219162580158432,0.9235880398671096
C3,2718,T2,0.8786044545253227,0.8791044776119403,0.8708133971291866,0.8863955119214586
C3,2718,T3,0.9259397379698132,0.9261194029850748,0.9222919937205653,0.9295874822190612
C3,2718,T4,0.9202199806433804,0.9205223880597015,0.9153081510934392,0.9251318101933216
C3,4242,T1,0.9272379853916264,0.9272388059701492,0.9269936353425684,0.9274823354406844
C3,4242,T2,0.8880185963027913,0.8884328358208955,0.8812077870480731,0.8948294055575097
C3,4242,T3,0.939105776043802,0.939179104477612,0.9369926555856204,0.9412188965019834
C3,4242,T4,0.9315948059251584,0.9317164179104478,0.9287105570705104,0.9344790547798066
C3,5678,T1,0.9298475816741568,0.9298507462686568,0.9303187546330616,0.9293764087152516
C3,5678,T2,0.8968934208501442,0.8970149253731343,0.893353941267388,0.9004329004329005
C3,5678,T3,0.952599737819563,0.9526119402985076,0.9518392112248768,0.953360264414249
C3,5678,T4,0.9510988073474362,0.9511194029850746,0.950095238095238,0.9521023765996344
C3,9001,T1,0.9208679786626556,0.9208955223880596,0.9193916349809886,0.9223443223443224
C3,9001,T2,0.8792178440043645,0.8798507462686567,0.8704746580852776,0.8879610299234516
C3,9001,T3,0.9331176885812242,0.9332089552238806,0.9306470360325456,0.9355883411299027
C3,9001,T4,0.9270591654010888,0.9272388059701492,0.9234393404004712,0.9306789904017064
C3,91011,T1,0.929079792197663,0.9291044776119404,0.9304029304029304,0.9277566539923956
C3,91011,T2,0.8954196481729657,0.8955223880597015,0.8921417565485362,0.8986975397973951
C3,91011,T3,0.947757004348756,0.9477611940298508,0.947289156626506,0.948224852071006
C3,91011,T4,0.954465338173132,0.9544776119402985,0.9537177541729894,0.9552129221732746
C4,1234,T1,0.6391315517242171,0.6720149253731343,0.7480653482373173,0.530197755211117
C4,1234,T2,0.4793060477538189,0.5708955223880597,0.6976866456361724,0.2609254498714653
C4,1234,T3,0.6388041911177385,0.6746268656716418,0.7525539160045402,0.5250544662309368
C4,1234,T4,0.5894774498660388,0.6395522388059701,0.7328539823008849,0.4461009174311927
C4,2718,T1,0.5151972178739066,0.5929104477611941,0.7092992272848387,0.3210952084629745
C4,2718,T2,0.3601272599166519,0.5123134328358209,0.6721845999498369,0.0480699198834668
C4,2718,T3,0.5128362883830192,0.5925373134328358,0.7098831030818279,0.3157894736842105
C4,2718,T4,0.4010427231801278,0.5313432835820896,0.6804071246819339,0.1216783216783216
C4,4242,T1,0.7525613086491973,0.7600746268656716,0.7956784238957737,0.7094441934026209
C4,4242,T2,0.596183415498363,0.6425373134328358,0.7329988851727982,0.4593679458239277
C4,4242,T3,0.8117048694889231,0.8164179104477612,0.8414948453608248,0.7819148936170213
C4,4242,T4,0.7679931266245419,0.7772388059701493,0.8143079315707621,0.7216783216783217
C4,5678,T1,0.6456536150430608,0.6757462686567164,0.7489164981219301,0.5423907319641916
C4,5678,T2,0.4343549735302313,0.5477611940298508,0.6876288659793814,0.1810810810810811
C4,5678,T3,0.6273724593594389,0.6652985074626866,0.7462517680339462,0.5084931506849315
C4,5678,T4,0.5385287845540481,0.607089552238806,0.716401831403178,0.360655737704918
C4,9001,T1,0.6456655578898225,0.6753731343283582,0.7482638888888888,0.5430672268907563
C4,9001,T2,0.4459590173398814,0.5537313432835821,0.6903158984981874,0.2016021361815754
C4,9001,T3,0.56128063733495,0.621268656716418,0.723508580768183,0.399052693901717
C4,9001,T4,0.4399549009153114,0.5507462686567164,0.6890495867768595,0.1908602150537634
C4,91011,T1,0.6861919724906622,0.7052238805970149,0.7634730538922155,0.6089108910891089
C4,91011,T2,0.4345963043284296,0.5473880597014925,0.6871292236265153,0.1820633850303439
C4,91011,T3,0.6979008565215461,0.7156716417910448,0.7711711711711712,0.6246305418719211
C4,91011,T4,0.6185228094148868,0.6567164179104478,0.7392290249433107,0.4978165938864629"""

df = pd.read_csv(io.StringIO(BASE_CONSOLIDADA))
ORDEM_C, ORDEM_T = ["C1", "C2", "C3", "C4"], ["T1", "T2", "T3", "T4"]
NOME_CONFIG = {"C1": "C1 - Full FT", "C2": "C2 - Freeze Lower (0-5)",
               "C3": "C3 - Freeze Upper (6-11)", "C4": "C4 - Frozen Encoder"}
NOME_TESTE = {"T1": "T1 - EN/Elec (base)", "T2": "T2 - EN/Beleza (Dominio)",
              "T3": "T3 - PT/Elec (Lingua prox.)", "T4": "T4 - PT/Beleza (Ambos)"}
sns.set_theme(style="whitegrid")

n_por_config = {c: int(df[df.config == c]["seed"].nunique()) for c in ORDEM_C}
print(f"{len(df)} medicoes | configs {sorted(df.config.unique())} | "
      f"cenarios {sorted(df.teste.unique())}")
print("seeds por configuracao (N dos testes):", n_por_config)

In [ ]:
### 4.1 F1-macro: media +/- desvio por configuracao x cenario

C1 e C2 praticamente empatam em todos os cenarios; C3 e consistentemente inferior; C4 (apenas *head*) desaba e, como veremos, com altissima variabilidade entre seeds.

In [ ]:
# Tabela 4.1 - F1-macro: media +/- desvio sobre as seeds (por config x cenario)
media = df.pivot_table(index="config", columns="teste", values="f1_macro",
                       aggfunc="mean").loc[ORDEM_C, ORDEM_T]
desvio = df.pivot_table(index="config", columns="teste", values="f1_macro",
                        aggfunc="std").loc[ORDEM_C, ORDEM_T]
tab1 = media.round(4).astype(str) + " +/- " + desvio.round(4).astype(str)
tab1.columns.name = "F1-macro (media +/- dp)"
tab1

### 3.1 Recipe de treino e *loop* dos modelos

A celula seguinte grava `src/train.py` (metricas, `TrainingArguments` e *early stopping*); a seguinte executa o *loop* de treino e avaliacao. Em GPU T4, cada conjunto de execucoes leva de uma a poucas horas, conforme o numero de seeds.

In [ ]:
### 4.2 Delta-shift: queda de F1-macro vs. baseline T1 (em pontos percentuais)

Convencao: `Delta = F1(T1) - F1(Tx)`. Valor **> 0 = perda** sob deslocamento; **< 0 = ganho** *zero-shot*. As colunas de **lingua proxima** sao negativas - o modelo vai **igual ou melhor** em Portugues, apesar de nunca ter treinado nesse idioma.

In [ ]:
# Tabela 4.2 - Delta-shift por configuracao (>0 = perda; <0 = ganho zero-shot)
deltas = pd.DataFrame({
    "Delta Dominio (T1-T2) pp":       (media["T1"] - media["T2"]) * 100,
    "Delta Lingua prox. (T1-T3) pp":  (media["T1"] - media["T3"]) * 100,
    "Delta Combinado (T1-T4) pp":     (media["T1"] - media["T4"]) * 100,
}).round(2)
deltas

> **Da execucao individual a base consolidada.** Este *loop* produz um `results.csv` por execucao. Para o estudo final, o procedimento foi repetido por tres integrantes do grupo com **seeds independentes**, e os CSVs foram combinados em uma unica base (6 a 8 seeds por configuracao). E essa base consolidada que a Etapa 4 analisa - uma **replicacao independente** que estressa as conclusoes muito alem das seeds iniciais.

---
## Etapa 4 - Analise estatistica consolidada (multi-seed)

Esta etapa e **auto-contida**: a base consolidada esta embutida na proxima celula, e toda a analise roda **sem GPU e sem Drive**. A base reune as execucoes independentes dos integrantes do grupo, com **6 a 8 seeds por configuracao** nos cenarios solidos (T1-T4). O numero maior de seeds em uma configuracao apenas aumenta a precisao da estimativa; os testes usados (Welch e Mann-Whitney) toleram N desigual.

Os resultados a seguir cobrem os deslocamentos que o desenho mede de forma limpa: **Domain Shift** (T2) e **Language Shift proximo** (T3, Ingles para Portugues). O *Language Shift* distante (Japones/Mandarim) e tratado na Etapa 9.

In [ ]:
# Etapa 4 - base consolidada embutida (self-contained, sem GPU/Drive).
# Reune as execucoes independentes dos integrantes do grupo (seeds distintas por
# pessoa): 6 a 8 seeds por configuracao, nos cenarios solidos T1-T4. E uma
# replicacao independente que estressa a robustez das conclusoes.
%matplotlib inline
import io
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

BASE_CONSOLIDADA = """\
config,seed,teste,f1_macro,accuracy,f1_negativo,f1_positivo
C1,13,T1,0.9466396439709718,0.946641791044776,0.9469781238413052,0.9463011641006384
C1,13,T2,0.9207807700952426,0.9208955223880596,0.9177657098525988,0.9237958303378864
C1,13,T3,0.957450150996813,0.9574626865671642,0.9567198177676538,0.958180484225972
C1,13,T4,0.9499618499513164,0.95,0.9485801995395242,0.9513435003631082
C1,888,T1,0.9350277322014552,0.9350746268656716,0.9367732558139537,0.933282208588957
C1,888,T2,0.9123016988656452,0.9123134328358208,0.9133161195130948,0.9112872782181956
C1,888,T3,0.95932708406576,0.9593283582089552,0.9590994371482175,0.9595547309833024
C1,888,T4,0.9582066276803118,0.9582089552238806,0.9578947368421052,0.9585185185185184
C1,1234,T1,0.941037387732368,0.941044776119403,0.940377358490566,0.9416974169741698
C1,1234,T2,0.9086376741615294,0.908955223880597,0.9032513877874704,0.9140239605355884
C1,1234,T3,0.9537032651234794,0.953731343283582,0.9525631216526396,0.954843408594319
C1,1234,T4,0.9480796133037838,0.948134328358209,0.9463941380640184,0.949765088543549
C1,2718,T1,0.9436565203014324,0.9436567164179104,0.9435514018691588,0.9437616387337058
C1,2718,T2,0.9177616437851738,0.917910447761194,0.9142634450506624,0.9212598425196852
C1,2718,T3,0.9525933998190148,0.9526119402985076,0.951655881233346,0.9535309184046836
C1,2718,T4,0.9499618499513164,0.95,0.9485801995395242,0.9513435003631082
C1,4242,T1,0.9455114332966252,0.9455223880597016,0.9462840323767476,0.9447388342165026
C1,4242,T2,0.9242144689851712,0.9242537313432836,0.9224894998090876,0.925939438161255
C1,4242,T3,0.9552028991079023,0.9552238805970148,0.954233409610984,0.956172388604821
C1,4242,T4,0.9507023411371238,0.9507462686567164,0.9492307692307692,0.9521739130434784
C1,5678,T1,0.9380149629048368,0.9380597014925374,0.9396802325581396,0.9363496932515336
C1,5678,T2,0.926491298931718,0.9264925373134328,0.9267930137495356,0.9261895841139004
C1,5678,T3,0.9567109935250296,0.9567164179104478,0.9562264150943396,0.9571955719557196
C1,5678,T4,0.9540926594630412,0.9541044776119404,0.9533560864618884,0.954829232464194
C1,9001,T1,0.9373084052185072,0.9373134328358208,0.9367469879518072,0.9378698224852072
C1,9001,T2,0.9116485655743428,0.9119402985074628,0.906571654790182,0.916725476358504
C1,9001,T3,0.9514492753623188,0.9514925373134328,0.95,0.9528985507246376
C1,9001,T4,0.9416998411973286,0.9417910447761194,0.9393939393939394,0.9440057430007178
C1,91011,T1,0.942163977792861,0.9421641791044776,0.9422718808193667,0.9420560747663552
C1,91011,T2,0.9174421277372176,0.9175373134328358,0.9146388567014292,0.9202453987730062
C1,91011,T3,0.9552118080391464,0.9552238805970148,0.9544764795144158,0.9559471365638766
C1,91011,T4,0.9473511078378553,0.9473880597014924,0.9459563050977386,0.9487459105779716
C2,1234,T1,0.925353013398577,0.9253731343283582,0.9265785609397944,0.9241274658573596
C2,1234,T2,0.9167420688406176,0.9167910447761194,0.9147227533460804,0.9187613843351548
C2,1234,T3,0.9514682119574304,0.9514925373134328,0.950381679389313,0.9525547445255474
C2,1234,T4,0.9495953627733752,0.9496268656716418,0.9483352468427096,0.9508554787040407
C2,2718,T1,0.939922353526624,0.9399253731343284,0.9394964299135664,0.9403482771396814
C2,2718,T2,0.9184987157660848,0.9186567164179104,0.9149102263856362,0.9220872051465332
C2,2718,T3,0.9540791016867056,0.9541044776119404,0.9529996178830722,0.955158585490339
C2,2718,T4,0.9480892215459552,0.948134328358209,0.946559015763168,0.9496194273287424
C2,4242,T1,0.9410383401173218,0.941044776119403,0.9416543574593796,0.940422322775264
C2,4242,T2,0.923434024420003,0.9235074626865672,0.9210627647285328,0.925805284111473
C2,4242,T3,0.9593188370133964,0.9593283582089552,0.9586964759378552,0.959941198088938
C2,4242,T4,0.9522148548503928,0.9522388059701492,0.9511450381679388,0.9532846715328468
C2,5678,T1,0.938801846192576,0.9388059701492536,0.9393042190969652,0.9382994732881866
C2,5678,T2,0.9227038439497116,0.9227611940298508,0.9205983889528192,0.9248092989466036
C2,5678,T3,0.9522148548503928,0.9522388059701492,0.9511450381679388,0.9532846715328468
C2,5678,T4,0.9492229359418806,0.9492537313432836,0.9479724560061208,0.9504734158776402
C2,9001,T1,0.9384324157925448,0.9384328358208956,0.9382716049382716,0.938593226646818
C2,9001,T2,0.9169375104707655,0.9171641791044776,0.9125984251968504,0.9212765957446808
C2,9001,T3,0.95108109143731,0.9511194029850746,0.9497120921305182,0.9524500907441016
C2,9001,T4,0.9458274522104309,0.9458955223880596,0.943907156673114,0.9477477477477476
C2,91011,T1,0.9402982082222512,0.9402985074626866,0.9401645474943904,0.9404318689501115
C2,91011,T2,0.918880484853557,0.9190298507462686,0.9153996101364522,0.922361359570662
C2,91011,T3,0.9533272709870196,0.9533582089552238,0.9521256223669092,0.9545289196071298
C2,91011,T4,0.948084533631124,0.948134328358209,0.9464767038891028,0.9496923633731452
C3,1234,T1,0.9231322307191496,0.923134328358209,0.9227306826706676,0.9235337787676318
C3,1234,T2,0.880539019257934,0.8809701492537313,0.8733624454148472,0.8877155931010208
C3,1234,T3,0.920708952620194,0.9208955223880596,0.9168627450980392,0.9245551601423488
C3,1234,T4,0.9164744398285156,0.9167910447761194,0.9113320079522864,0.9216168717047452
C3,2718,T1,0.9227521489414764,0.9227611940298508,0.9219162580158432,0.9235880398671096
C3,2718,T2,0.8786044545253227,0.8791044776119403,0.8708133971291866,0.8863955119214586
C3,2718,T3,0.9259397379698132,0.9261194029850748,0.9222919937205653,0.9295874822190612
C3,2718,T4,0.9202199806433804,0.9205223880597015,0.9153081510934392,0.9251318101933216
C3,4242,T1,0.9272379853916264,0.9272388059701492,0.9269936353425684,0.9274823354406844
C3,4242,T2,0.8880185963027913,0.8884328358208955,0.8812077870480731,0.8948294055575097
C3,4242,T3,0.939105776043802,0.939179104477612,0.9369926555856204,0.9412188965019834
C3,4242,T4,0.9315948059251584,0.9317164179104478,0.9287105570705104,0.9344790547798066
C3,5678,T1,0.9298475816741568,0.9298507462686568,0.9303187546330616,0.9293764087152516
C3,5678,T2,0.8968934208501442,0.8970149253731343,0.893353941267388,0.9004329004329005
C3,5678,T3,0.952599737819563,0.9526119402985076,0.9518392112248768,0.953360264414249
C3,5678,T4,0.9510988073474362,0.9511194029850746,0.950095238095238,0.9521023765996344
C3,9001,T1,0.9208679786626556,0.9208955223880596,0.9193916349809886,0.9223443223443224
C3,9001,T2,0.8792178440043645,0.8798507462686567,0.8704746580852776,0.8879610299234516
C3,9001,T3,0.9331176885812242,0.9332089552238806,0.9306470360325456,0.9355883411299027
C3,9001,T4,0.9270591654010888,0.9272388059701492,0.9234393404004712,0.9306789904017064
C3,91011,T1,0.929079792197663,0.9291044776119404,0.9304029304029304,0.9277566539923956
C3,91011,T2,0.8954196481729657,0.8955223880597015,0.8921417565485362,0.8986975397973951
C3,91011,T3,0.947757004348756,0.9477611940298508,0.947289156626506,0.948224852071006
C3,91011,T4,0.954465338173132,0.9544776119402985,0.9537177541729894,0.9552129221732746
C4,1234,T1,0.6391315517242171,0.6720149253731343,0.7480653482373173,0.530197755211117
C4,1234,T2,0.4793060477538189,0.5708955223880597,0.6976866456361724,0.2609254498714653
C4,1234,T3,0.6388041911177385,0.6746268656716418,0.7525539160045402,0.5250544662309368
C4,1234,T4,0.5894774498660388,0.6395522388059701,0.7328539823008849,0.4461009174311927
C4,2718,T1,0.5151972178739066,0.5929104477611941,0.7092992272848387,0.3210952084629745
C4,2718,T2,0.3601272599166519,0.5123134328358209,0.6721845999498369,0.0480699198834668
C4,2718,T3,0.5128362883830192,0.5925373134328358,0.7098831030818279,0.3157894736842105
C4,2718,T4,0.4010427231801278,0.5313432835820896,0.6804071246819339,0.1216783216783216
C4,4242,T1,0.7525613086491973,0.7600746268656716,0.7956784238957737,0.7094441934026209
C4,4242,T2,0.596183415498363,0.6425373134328358,0.7329988851727982,0.4593679458239277
C4,4242,T3,0.8117048694889231,0.8164179104477612,0.8414948453608248,0.7819148936170213
C4,4242,T4,0.7679931266245419,0.7772388059701493,0.8143079315707621,0.7216783216783217
C4,5678,T1,0.6456536150430608,0.6757462686567164,0.7489164981219301,0.5423907319641916
C4,5678,T2,0.4343549735302313,0.5477611940298508,0.6876288659793814,0.1810810810810811
C4,5678,T3,0.6273724593594389,0.6652985074626866,0.7462517680339462,0.5084931506849315
C4,5678,T4,0.5385287845540481,0.607089552238806,0.716401831403178,0.360655737704918
C4,9001,T1,0.6456655578898225,0.6753731343283582,0.7482638888888888,0.5430672268907563
C4,9001,T2,0.4459590173398814,0.5537313432835821,0.6903158984981874,0.2016021361815754
C4,9001,T3,0.56128063733495,0.621268656716418,0.723508580768183,0.399052693901717
C4,9001,T4,0.4399549009153114,0.5507462686567164,0.6890495867768595,0.1908602150537634
C4,91011,T1,0.6861919724906622,0.7052238805970149,0.7634730538922155,0.6089108910891089
C4,91011,T2,0.4345963043284296,0.5473880597014925,0.6871292236265153,0.1820633850303439
C4,91011,T3,0.6979008565215461,0.7156716417910448,0.7711711711711712,0.6246305418719211
C4,91011,T4,0.6185228094148868,0.6567164179104478,0.7392290249433107,0.4978165938864629"""

df = pd.read_csv(io.StringIO(BASE_CONSOLIDADA))
ORDEM_C, ORDEM_T = ["C1", "C2", "C3", "C4"], ["T1", "T2", "T3", "T4"]
NOME_CONFIG = {"C1": "C1 - Full FT", "C2": "C2 - Freeze Lower (0-5)",
               "C3": "C3 - Freeze Upper (6-11)", "C4": "C4 - Frozen Encoder"}
NOME_TESTE = {"T1": "T1 - EN/Elec (base)", "T2": "T2 - EN/Beleza (Dominio)",
              "T3": "T3 - PT/Elec (Lingua prox.)", "T4": "T4 - PT/Beleza (Ambos)"}
sns.set_theme(style="whitegrid")

n_por_config = {c: int(df[df.config == c]["seed"].nunique()) for c in ORDEM_C}
print(f"{len(df)} medicoes | configs {sorted(df.config.unique())} | "
      f"cenarios {sorted(df.teste.unique())}")
print("seeds por configuracao (N dos testes):", n_por_config)

### 4.1 F1-macro: media +/- desvio por configuracao x cenario

C1 e C2 praticamente empatam em todos os cenarios; C3 e consistentemente inferior; C4 (apenas *head*) desaba e, como veremos, com altissima variabilidade entre seeds.

In [ ]:
# Tabela 4.1 - F1-macro: media +/- desvio sobre as seeds (por config x cenario)
media = df.pivot_table(index="config", columns="teste", values="f1_macro",
                       aggfunc="mean").loc[ORDEM_C, ORDEM_T]
desvio = df.pivot_table(index="config", columns="teste", values="f1_macro",
                        aggfunc="std").loc[ORDEM_C, ORDEM_T]
tab1 = media.round(4).astype(str) + " +/- " + desvio.round(4).astype(str)
tab1.columns.name = "F1-macro (media +/- dp)"
tab1

### 4.2 Delta-shift: queda de F1-macro vs. baseline T1 (em pontos percentuais)

Convencao: `Delta = F1(T1) - F1(Tx)`. Valor **> 0 = perda** sob deslocamento; **< 0 = ganho** *zero-shot*. As colunas de **lingua proxima** sao negativas - o modelo vai **igual ou melhor** em Portugues, apesar de nunca ter treinado nesse idioma.

In [ ]:
# Tabela 4.2 - Delta-shift por configuracao (>0 = perda; <0 = ganho zero-shot)
deltas = pd.DataFrame({
    "Delta Dominio (T1-T2) pp":       (media["T1"] - media["T2"]) * 100,
    "Delta Lingua prox. (T1-T3) pp":  (media["T1"] - media["T3"]) * 100,
    "Delta Combinado (T1-T4) pp":     (media["T1"] - media["T4"]) * 100,
}).round(2)
deltas

### 4.3 Significancia estatistica vs. baseline C1

Com 6-8 seeds, o Mann-Whitney U deixa de saturar e desce a p ~ 0.001, e os efeitos ficam nitidos. Nenhuma configuracao congelada **supera** a C1 de forma significativa; C3 e C4 sao significativamente **piores** em todos os cenarios.

In [ ]:
# Tabela 4.3 - significancia vs. baseline C1 (Welch t + Mann-Whitney U + Cohen d)
def cohen_d(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    na, nb = len(a), len(b)
    sp2 = ((na - 1) * a.var(ddof=1) + (nb - 1) * b.var(ddof=1)) / (na + nb - 2)
    return 0.0 if sp2 == 0 else (a.mean() - b.mean()) / np.sqrt(sp2)

def amostras(cfg, teste):
    return df[(df.config == cfg) & (df.teste == teste)].sort_values("seed")["f1_macro"].to_numpy()

comparacoes = [
    ("T2", "C2", "Dominio - Freeze Lower vs Full"),
    ("T2", "C3", "Dominio - Freeze Upper vs Full (aposta H2)"),
    ("T2", "C4", "Dominio - Frozen Encoder vs Full"),
    ("T3", "C2", "Lingua prox. - Freeze Lower vs Full (aposta H1)"),
    ("T3", "C3", "Lingua prox. - Freeze Upper vs Full"),
    ("T3", "C4", "Lingua prox. - Frozen Encoder vs Full"),
]
rows = []
for teste, cfg, desc in comparacoes:
    base, alt = amostras("C1", teste), amostras(cfg, teste)
    _, p_t = stats.ttest_ind(alt, base, equal_var=False)
    _, p_u = stats.mannwhitneyu(alt, base, alternative="two-sided")
    rows.append({"cenario": teste, "comparacao": f"{cfg} vs C1", "descricao": desc,
                 "Delta pp": round((alt.mean() - base.mean()) * 100, 2),
                 "p (Welch)": round(float(p_t), 4), "p (MWU)": round(float(p_u), 3),
                 "Cohen d": round(cohen_d(alt, base), 2),
                 "sig. p<0.10": "sim" if p_t < 0.10 else "nao"})
pd.DataFrame(rows)

### 4.4 Veredito das hipoteses e robustez sob replicacao

Criterio de confirmacao: mitigacao real exige **Delta >= +3 pp E p < 0.10**. A celula abaixo aplica o criterio a H1 e H2 e, em seguida, testa se um achado marginal de analises com poucas seeds (a ideia de que a C2 *protegeria* contra o Domain Shift) **se sustenta** com a base maior - um teste direto da robustez das conclusoes.

In [ ]:
# Veredito formal (criterio: Delta >= +3 pp E p < 0.10) e teste de robustez
for h, cfg, teste, alvo in [("H1", "C2", "T3", "Language Shift proximo"),
                            ("H2", "C3", "T2", "Domain Shift")]:
    alt, base = amostras(cfg, teste), amostras("C1", teste)
    d = (alt.mean() - base.mean()) * 100
    _, p = stats.ttest_ind(alt, base, equal_var=False)
    shift = (media.loc["C1", "T1"] - media.loc["C1", teste]) * 100
    ok = (d >= 3.0) and (p < 0.10)
    print(f"{h} - {cfg} mitiga {alvo}?")
    print(f"   shift na baseline C1 (T1-{teste}) = {shift:+.2f} pp"
          f"  ->  {'ha queda' if shift > 0 else 'NAO ha queda (ganho zero-shot)'}")
    print(f"   Delta({cfg}-C1) em {teste} = {d:+.2f} pp | p = {p:.4f}")
    print(f"   VEREDITO: {'CONFIRMADA' if ok else 'REFUTADA'}\n")

# Robustez: o achado marginal "C2 protege contra Domain Shift" sobrevive a mais seeds?
alt, base = amostras("C2", "T2"), amostras("C1", "T2")
d = (alt.mean() - base.mean()) * 100
_, p = stats.ttest_ind(alt, base, equal_var=False)
print("TESTE DE ROBUSTEZ - C2 mitiga Domain Shift (achado marginal de poucas seeds)?")
print(f"   Delta(C2-C1) em T2 = {d:+.2f} pp | p = {p:.4f}"
      f"  ->  {'significativo' if p < 0.10 else 'NAO significativo'} a p<0.10")
print("   Conclusao: com 6-8 seeds independentes o efeito desaparece - C2 e equivalente a C1.")

# Estabilidade entre seeds por configuracao (motiva a leitura sobre a C4)
print("\nDesvio medio de F1-macro entre seeds (estabilidade):")
for c in ORDEM_C:
    s = float(np.mean([df[(df.config == c) & (df.teste == t)]["f1_macro"].std() for t in ORDEM_T]))
    print(f"   {c}: {s:.4f}")

In [ ]:
# (1) Heatmap F1-macro: configuracao x cenario
hm = media.copy()
hm.index = [NOME_CONFIG[c] for c in hm.index]
hm.columns = [NOME_TESTE[t] for t in hm.columns]
plt.figure(figsize=(10, 6))
ax = sns.heatmap(hm, annot=True, fmt=".3f", cmap="viridis", linewidths=0.5,
                 cbar_kws={"label": "F1-macro (media das seeds)"})
ax.set_title("F1-macro por Configuracao x Cenario (base consolidada)", weight="bold", pad=12)
ax.set_xlabel(""); ax.set_ylabel("")
plt.xticks(rotation=20, ha="right"); plt.yticks(rotation=0)
plt.tight_layout(); plt.show()

In [ ]:
# (2) Barplot dos Delta-shift (Dominio T1-T2 vs Lingua proxima T1-T3) com barras de erro
def deltas_seed(cfg, ta, tb):
    a = df[(df.config == cfg) & (df.teste == ta)].sort_values("seed")
    b = df[(df.config == cfg) & (df.teste == tb)].sort_values("seed")
    m = a.merge(b, on="seed", suffixes=("_a", "_b"))
    return (m["f1_macro_a"].to_numpy() - m["f1_macro_b"].to_numpy()) * 100

dom_m, dom_s, lin_m, lin_s = [], [], [], []
for c in ORDEM_C:
    dd, dl = deltas_seed(c, "T1", "T2"), deltas_seed(c, "T1", "T3")
    dom_m.append(dd.mean()); dom_s.append(dd.std(ddof=1))
    lin_m.append(dl.mean()); lin_s.append(dl.std(ddof=1))

x = np.arange(len(ORDEM_C)); w = 0.38
plt.figure(figsize=(10, 6))
plt.bar(x - w / 2, dom_m, w, yerr=dom_s, capsize=5, label="Domain Shift (T1-T2)", color="#d1495b")
plt.bar(x + w / 2, lin_m, w, yerr=lin_s, capsize=5, label="Language Shift proximo (T1-T3)", color="#30638e")
plt.axhline(0, color="black", lw=0.8)
plt.xticks(x, ORDEM_C); plt.ylabel("Delta F1-macro vs T1 (pp)")
plt.title("Delta-shift por configuracao  (>0 = perda;  <0 = ganho zero-shot)", weight="bold")
plt.legend(); plt.tight_layout(); plt.show()

### 4.5 Leitura dos resultados

- **H1 (Freeze Lower mitiga Language Shift) - refutada por ausencia do fenomeno.** No par Ingles->Portugues nao ha queda a mitigar: a baseline ja **ganha** em Portugues (*zero-shot cross-lingual*). C2 nao altera isso.
- **H2 (Freeze Upper mitiga Domain Shift) - refutada e invertida.** Congelar o topo (C3) **piora** o desempenho em Beleza, com efeito grande e estatisticamente forte. O resultado e o oposto do previsto.
- **C2 e equivalente a C1, e esse e o seu valor.** A C2 nunca e significativamente melhor nem pior que o *fine-tuning* completo - mas treina cerca de metade dos parametros. O ganho da C2 e **eficiencia**, nao regularizacao.
- **C4 (Frozen Encoder) e inviavel e instavel.** Alem de fraco, seu desempenho varia enormemente entre seeds: *probing* linear do XLM-RoBERTa nesta tarefa e uma loteria de inicializacao.

> **Escopo.** A conclusao sobre *Language Shift* e restrita ao par **proximo** Ingles-Portugues (duas linguas indo-europeias, de alta cobertura no pre-treino) e esta confundida com a assimetria de qualidade dos conjuntos (Portugues e *ground-truth*; Ingles e filtro ruidoso). A pergunta que **decide** a tese - se a robustez se mantem em uma lingua **tipologicamente distante** - e o objeto da Etapa 9.

---
## Etapa 9 - Proximo passo: *Language Shift* em lingua distante

Esta e a pergunta que motivou todo o desenho e a continuidade natural do trabalho: a ausencia de *Language Shift* observada em Ingles->Portugues **se mantem em uma lingua tipologicamente distante**, como Japones ou Mandarim? Se a queda aparecer la, o *Language Shift* existe e estava apenas **oculto** pela proximidade Ingles-Portugues; se nao aparecer, a robustez multilingue do XLM-RoBERTa se confirma como achado forte.

O experimento usa o dataset **MARC** (*Multilingual Amazon Reviews Corpus*, espelho `mteb/amazon_reviews_multi`), construindo:

- **T5 (JA)** e **T6 (ZH)** - celulas de lingua distante;
- **T7 (EN-ancora)** - mesma fonte e composicao de dominio que T5/T6, para que a comparacao **T7 -> T5/T6** isole a *distancia linguistica* (e nao o dominio).

A celula a seguir traz o carregador das celulas do MARC, pronto para a proxima execucao em GPU. Ele carrega cada lingua de forma **posicional** e **confere** a lingua efetivamente recebida - uma boa pratica que garante que JA, ZH e EN sejam conjuntos distintos antes da avaliacao. Os modelos C1-C4 ja treinados (Etapa 3) podem ser **reaproveitados**: basta rodar a avaliacao nas novas celulas, sem novo treino.

In [ ]:
# Carregador das celulas do MARC para a proxima execucao (lingua distante JA/ZH).
# NAO roda aqui (exige Internet/GPU); e a celula que abre a Etapa 9.
# Boas praticas embutidas:
#   (1) carrega a config de lingua de forma POSICIONAL: load_dataset(repo, lang, ...);
#   (2) CONFERE a lingua efetivamente carregada (quando ha coluna de idioma);
#   (3) sanidade: os DataFrames de JA e ZH devem ser conjuntos distintos.
from datasets import load_dataset

def carregar_marc(lang):
    """Carrega o split de teste do MARC para `lang` e confirma a lingua recebida."""
    erros = []
    for repo in ["mteb/amazon_reviews_multi", "amazon_reviews_multi"]:
        try:
            ds = load_dataset(repo, lang, split="test")   # config POSICIONAL de lingua
        except Exception as e:
            erros.append(f"{repo} name={lang}: {str(e)[:90]}")
            continue
        col = next((c for c in ds.column_names if c.lower() in ("language", "lang")), None)
        if col is not None:
            langs = set(map(str, set(ds[col])))
            assert langs == {lang}, f"{repo} devolveu linguas {langs}, esperava {{{lang}}}"
        return ds, repo, {"name": lang}
    raise RuntimeError("Falha ao carregar MARC por lingua:\n" + "\n".join(erros))

# Uso na proxima execucao (apos construir as celulas):
#   ds_ja, *_ = carregar_marc("ja"); df_ja_raw = pd.DataFrame(ds_ja)
#   ds_zh, *_ = carregar_marc("zh"); df_zh_raw = pd.DataFrame(ds_zh)
#   ds_en, *_ = carregar_marc("en"); df_en_raw = pd.DataFrame(ds_en)   # ancora T7
#   assert not df_ja_raw.equals(df_zh_raw), "JA e ZH devem ser conjuntos distintos"
# Em seguida, reaproveite os modelos C1-C4 ja treinados e avalie em T5/T6/T7.
print("Carregador MARC definido. Rode esta secao em GPU para o proximo passo (lingua distante).")

---
## Conclusoes

1. **Robustez sob replicacao.** Com 6-8 seeds independentes, as conclusoes se mostram estaveis: as configuracoes viaveis (C1, C2, C3) reproduzem os resultados entre execucoes independentes dentro de menos de 1 ponto percentual.
2. **H1 e H2 refutadas.** Nao ha *Language Shift* a mitigar no par proximo Ingles-Portugues; e congelar o topo (C3) **piora** o Domain Shift, em vez de mitiga-lo (efeito grande, p < 0.001).
3. **C2 vale pela eficiencia.** Congelar a base (C2) iguala o *fine-tuning* completo treinando metade dos parametros - sem, contudo, oferecer a protecao de dominio que analises preliminares sugeriam.
4. **C4 e um piso inviavel e instavel.** *Probing* linear do encoder e fraco e fortemente dependente da seed.
5. **A fronteira esta definida.** O teste decisivo - lingua tipologicamente distante - esta desenhado e pronto para execucao (Etapa 9). E o proximo passo natural do trabalho.

O ciclo experimental (dados -> arquitetura -> treino -> analise -> replicacao) esta fechado e e reproduzivel por este notebook.